In [ ]:
#Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Set Plot Aesthetics
sns.set_theme(style="whitegrid")

In [ ]:
# Step 1 === Load the Dataset===
df_wine = load_wine()
X = pd.DataFrame(df_wine.data, columns=df_wine.feature_names)
y = df_wine.target

print("== Original Dataset Shape ==")
print(f"Sample: {X.shape[0]}, Original Features: {X.shape[1]}\n")

In [ ]:
# Step 2 == Preprocessing (Feature Scaling) ====
#PCA is very very sensitive to scaling. You must scale!!!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Step 3 === Fit PCA to examine the variance profile ===
# We are instantiating PCA without restricting components
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Calculate the variance characteristics
explained_variance_ratio = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

In [ ]:
# Step 4 === Visualize Optimal Components ===
fig, ax1 =plt.subplots(figsize=(10,6))

# Individual Variance Bar Chart
ax1.bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, alpha=0.7, color='b', label='individual variance')
ax1.set_xlabel( 'Principal Component Index')
ax1.set_ylabel('Variance Ratio Explained (individual)', color='b')
ax1.tick_params(axis='y', labelcolor='b')

# # Cumulative Variance Step Line
ax2 = ax1.twinx()
# ax2.plot(range(1, len(cumulative_variance), +1), cumulative_variance, marker='o', color='g', label='Cumulative Variance')
# ax2.set_ylabel('Cumulative Variance Ratio Explained', color='g'
#ax2.tick_params(axis='y', labelcolor='g')


# Cumulative Variance Bar Chart
ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', color='g', label='Cumulative Variance')
ax2.set_xlabel('Principal Component Index')
ax2.set_ylabel('Cumulative Variance Ratio Explained', color='g' )
ax2.tick_params(axis='y', labelcolor='g')


In [ ]:
# Find the exact component index that crosses our 95% variance target
optimal_component = np.argmax(cumulative_variance >= 0.95) +1
print(f' -> {optimal_component} components are needed to presere 95% total variance')

In [ ]:
# Rerun PCA with the optimal number of components
pca_10d = PCA(n_components=10, random_state=42)
X_pca_10d = pca_10d.fit_transform(X_scaled)

In [ ]:
# Create a clean dataframe holding our low dimension data
pca_column =['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
# pca_column = [f'PC[i]' for i in range(1, 11)]
df_pca = pd.DataFrame(data=X_pca_10d, columns=pca_column)
df_pca['Target'] = y
df_pca['Target_Name'] = df_pca['Target'].map(lambda x: df_wine.target_names[x])


In [ ]:
X.head()

In [ ]:
df_pca.head()

In [ ]:
# == Step 6: Visualize the reduced cluster splitting ==
plt.figure(figsize=(10,6))
sns.scatterplot(
    x='PC1',
    y='PC10',
    hue='Target_Name',
    data=df_pca,
    palette='bright',
    s=100,
    edgecolor='black',
    alpha=0.9
)

plt.title('Wine Dataset - 2D PCA')
plt.xlabel(f'Principal Component 1 ({explained_variance_ratio[0]*100:.1f}%) Variance)')
plt.ylabel(f'principal Component 10 ({explained_variance_ratio[9]*100:.1f}%) Variance)')
plt.legend(title='Wine Classes')
plt.tight_layout()
plt.show()

In [ ]:
# == Step 7: Interpret Feature Loadings
loadings = pd.DataFrame(pca_10d.components_.T, columns=pca_column, index=df_wine.feature_names)
print("Top Features Loading for PC1")
print(loadings['PC1'].sort_values(ascending=False).head())